1. Import ResNet


In [2]:
import torch
import torch.nn as nn 
import torchvision.models as models 

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Pytorch:", torch.__version__)
print("Device:", device)

Pytorch: 2.14.0
Device: mps


2. Load pretrained ResNet - 50

In [3]:
weights = models.ResNet50_Weights.DEFAULT
resnet = models.resnet50(
    weights = weights
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/vivekkumar/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:08<00:00, 11.4MB/s]


In [4]:
print(resnet)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [5]:
print(resnet.fc)

Linear(in_features=2048, out_features=1000, bias=True)


In [6]:
print("ResNet feature dimension:", resnet.fc.in_features)
print("Original output classes:", resnet.fc.out_features)

ResNet feature dimension: 2048
Original output classes: 1000


### Select Image Encoder

- Selected pretrained ResNet-50
- Loaded ImageNet pretrained weights
- Verified PyTorch and MPS device
- Verified original classification head
- ResNet feature dimension: 2048
- Original ImageNet classes: 1000

3. Replace the Classification Head

In [9]:
import torch 
import torch.nn as nn
import torchvision.models as models

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
NUM_CLASSES = 20
weights = models.ResNet50_Weights.DEFAULT
resnet = models.resnet50(weights=weights)

print("Original classifier:")
print(resnet.fc)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
print("\nModified classifier:")
print(resnet.fc)

Original classifier:
Linear(in_features=2048, out_features=1000, bias=True)

Modified classifier:
Linear(in_features=2048, out_features=20, bias=True)


4. Move Model to MPS

In [10]:
resnet = resnet.to(device)
print("Model device:", next(resnet.parameters()).device)

Model device: mps:0


5. Verify the Model Output

In [13]:
%run ./02_Data_Preparation.ipynb

Device: mps
Train: (29901, 6)
Validation: (6435, 6)
Test: (6459, 6)
../data/raw/fashion-product-images-small/images/15970.jpg
Exists: True
Image size: (60, 80)
Image mode: RGB
Image mode: RGB
Tensor shape: torch.Size([3, 224, 224])
Tensor dtype: torch.float32
Tenosr min: -2.0836544036865234
Tensor max: 2.640000104904175
Dataset size: 29901
Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: tensor(17)
Label dtype: torch.int64
index 0: shape=torch.Size([3, 224, 224]), label=17
index 100: shape=torch.Size([3, 224, 224]), label=17
index 1000: shape=torch.Size([3, 224, 224]), label=13
index 5000: shape=torch.Size([3, 224, 224]), label=17
index 10000: shape=torch.Size([3, 224, 224]), label=0
Train datasets: 29901
Validation datasets: 6435
Test dataset: 6459
Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Images dtype: torch.float32
Label dtype: torch.int64
Labels: tensor([ 5, 15, 17,  7, 15, 17, 15, 15,  7,  0, 19,  4, 17,  8, 17,  8,  2, 15,


Text: Turtle Check Men Navy Blue Shirt
Input IDs shape: torch.Size([1, 32])
Attention mask shape: torch.Size([1, 32])
['[CLS]', 'turtle', 'check', 'men', 'navy', 'blue', 'shirt', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])
Orignal : Turtle Check Men Navy Blue Shirt
Decoded : turtle check men navy blue shirt
Input IDs shape: torch.Size([8, 32])
Attention mask shape: torch.Size([8, 32])
Maximum token length:  30
Texts with > 32 tokens: 0
Multimodal train dataset: 29901
dict_keys(['image', 'input_ids', 'attention_mask', 'label'])
Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Input IDs shape: torch.Size([32])
Input IDs dtype: torch.int64
Attention mask shape: torch.Size([32])
At

In [17]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = resnet(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", labels.shape)

Input shape: torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 20])
Labels shape: torch.Size([32])


6. Freeze the Backbone

In [18]:
for param in resnet.parameters():
    param.requires_grad = False

for param in resnet.fc.parameters():
    param.requires_grad = True

In [19]:
trainable_params = sum(
    p.numel() for p in resnet.parameters()
    if p.requires_grad
)
total_params = sum(
    p.numel() for p in resnet.parameters()
)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 23549012
Trainable parameters: 40980


### Modify ResNet-50 for 20 Classes

- Loaded ImageNet-pretrained ResNet-50
- Replaced 1000-class ImageNet classifier with 20-class classifier
- Verified input shape: [32, 3, 224, 224]
- Verified output shape: [32, 20]
- Moved model to MPS
- Frozen pretrained ResNet-50 backbone
- Kept new classification head trainable
- Total parameters: 23,549,012
- Trainable parameters: 40,980

7. Cross-Entropy Loss

In [20]:
criterion = nn.CrossEntropyLoss()
print("Loss function:", criterion)

Loss function: CrossEntropyLoss()


8. Optimizer

In [23]:
optimizer = torch.optim.Adam(
    resnet.fc.parameters(),
    lr = 0.001
)
print("Optimizer:", optimizer)

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


9. Learning Rate Scheduler

In [25]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience = 2
)
print("Scheduler:", scheduler)

Scheduler: <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x143921810>


In [26]:
print("Trainable parameters:")

for name, param in resnet.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

Trainable parameters:
fc.weight torch.Size([20, 2048])
fc.bias torch.Size([20])


In [27]:
optimizer_params = sum(
    p.numel()
    for group in optimizer.param_groups
    for p in group["params"]
)
print("Parameters passed to optimizer:", optimizer_params)

Parameters passed to optimizer: 40980


### Configure Image Model Training

- Loss function: CrossEntropyLoss
- Optimizer: Adam
- Learning rate: 0.001
- Scheduler: ReduceLROnPlateau
- Scheduler factor: 0.1
- Scheduler patience: 2 epochs
- ResNet-50 backbone: Frozen
- Classification head: Trainable
- Optimizer parameters: 40,980